In [0]:
data = [
    (1, "A"),
    (1, "B"),
    (1, "B"),
    (1, "A"),
    (2, "C"),
    (2, "B"),
    (2, "B"),
    (3, "D"),
    (3, "A"),
    (3, "D")
]

columns = ["key", "value"]

df = spark.createDataFrame(data, columns)

df.createOrReplaceTempView("df_v")

spark.sql(
        """
        SELECT  key, aggregated_values
FROM (
        SELECT 
            key,
            concat_ws(
                '',
                collect_list(value)
            ) AS aggregated_values
        FROM df_v
        GROUP BY key
        ) x
WHERE aggregated_values = reverse(aggregated_values)
        """
    ).show()

from pyspark.sql.functions import concat_ws,collect_list,reverse,col
df_agg=df.groupBy("key").agg(concat_ws('',collect_list("value")).alias("word"))
df_agg.filter(
        col("word") == reverse(col("word"))
    ).show()